# Cookie Cats A/B Test: Does Moving the First Gate Hurt Retention?

**Experiment.** In the mobile puzzle game Cookie Cats, players who reach a "gate" level must wait or pay before continuing. New players were randomly assigned at install to the original design (`gate_30`, control: first gate at level 30) or a variant (`gate_40`, treatment: first gate moved to level 40).

**Question.** Does moving the first gate later change player retention?

**Data.** 90,189 players from the Kaggle dataset "Mobile Games A/B Testing - Cookie Cats" (originally from DataCamp). The raw CSV is not included in this repository; see `../data/README.md`.

**Decision rules, fixed before viewing results.**
- Sample ratio mismatch (SRM) threshold: p < 0.001, assuming a 50/50 design.
- Primary metric: D7 retention. Secondary: D1 retention. Guardrail: game rounds in the first 14 days.
- Two-sided α = 0.05. Differences are reported as `gate_40 − gate_30`.

In [1]:
import pandas as pd
import scipy
import statsmodels

print(pd.__version__, scipy.__version__, statsmodels.__version__)

2.0.3 1.11.1 0.14.0


## 1. Load data

In [2]:
df = pd.read_csv("../data/cookie_cats.csv")
print(df.shape)
df.head()

(90189, 5)


,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True


## 2. Data validation

Checks for duplicates, missing values, group labels, extreme values, and how the retention flags are defined.

- `userid` is unique and no column has missing values.
- Only two labels exist: `gate_30` (44,700) and `gate_40` (45,489).
- 3,599 players were inactive on day 1 but active on day 7, so `retention_7` means activity *on* day 7, not continuous activity through day 7.
- One `gate_30` player (`userid` 6390605) logged 49,854 rounds in 14 days (16.8 times the next highest). This player is kept for retention metrics and excluded from game-round comparisons.

In [3]:
print("unique userid:", df["userid"].nunique())
print(df.isna().sum())
print(df["version"].value_counts())

unique userid: 90189
userid            0
version           0
sum_gamerounds    0
retention_1       0
retention_7       0
dtype: int64
version
gate_40    45489
gate_30    44700
Name: count, dtype: int64


In [4]:
print(df["sum_gamerounds"].describe())
print(df["sum_gamerounds"].nlargest(5))
print("zero rounds:", (df["sum_gamerounds"] == 0).sum())

count    90189.000000
mean        51.872457
std        195.050858
min          0.000000
25%          5.000000
50%         16.000000
75%         51.000000
max      49854.000000
Name: sum_gamerounds, dtype: float64
57702    49854
7912      2961
29417     2640
43671     2438
48188     2294
Name: sum_gamerounds, dtype: int64
zero rounds: 3994


In [5]:
pd.crosstab(df["retention_1"], df["retention_7"])

retention_7,False,True
retention_1,,
False,46437,3599
True,26971,13182


In [6]:
df.loc[57702]

userid            6390605
version           gate_30
sum_gamerounds      49854
retention_1         False
retention_7          True
Name: 57702, dtype: object

## 3. Sample ratio mismatch check

Tests whether the 44,700 vs 45,489 split is consistent with a 50/50 allocation. A failed check would mean the assignment or logging is broken, and the treatment effect could not be trusted.

**Result:** χ² = 6.902, p = 0.0086. This does not cross the pre-set p < 0.001 threshold, so the analysis proceeds. The result is borderline, and the intended allocation is not documented, so it is reported as a caveat.

In [7]:
from scipy.stats import chisquare

observed = df["version"].value_counts().sort_index()
expected = [len(df) / 2, len(df) / 2]

stat, p_value = chisquare(f_obs=observed, f_exp=expected)
print(observed)
print("chi-square:", round(stat, 3), " p-value:", round(p_value, 5))

version
gate_30    44700
gate_40    45489
Name: count, dtype: int64
chi-square: 6.902  p-value: 0.00861


## 4. Primary and secondary metrics: D7 and D1 retention

Retention rates by group, then a two-proportion z-test for each metric.

**Result:** D7 fell from 19.02% to 18.20% (−0.82 pp, p = 0.0016), a statistically significant drop. D1 fell from 44.82% to 44.23% (−0.59 pp, p = 0.074), not significant at α = 0.05.

In [8]:
df.groupby("version")[["retention_1", "retention_7"]].mean()

,retention_1,retention_7
version,,
gate_30,0.448188,0.190201
gate_40,0.442283,0.182000


In [9]:
from statsmodels.stats.proportion import proportions_ztest

n = df["version"].value_counts().sort_index()

for col in ["retention_1", "retention_7"]:
    retained = df.groupby("version")[col].sum().sort_index()
    rate = retained / n
    z, p = proportions_ztest(count=retained, nobs=n)
    diff_pp = (rate["gate_40"] - rate["gate_30"]) * 100
    print(col, "| gate_40 - gate_30:", round(diff_pp, 2), "pp | z:", round(z, 3), "| p:", round(p, 5))

retention_1 | gate_40 - gate_30: -0.59 pp | z: 1.784 | p: 0.07441
retention_7 | gate_40 - gate_30: -0.82 pp | z: 3.164 | p: 0.00155


## 5. Effect size: bootstrap confidence interval for D7

Resamples players with replacement *within each group* 2,000 times (seed 42) to estimate how much the D7 difference could vary from sampling alone.

**Result:** −0.83 pp, 95% CI −1.35 to −0.34 pp. The interval excludes zero: the drop is real, and plausibly between about 0.3 and 1.4 percentage points.

In [10]:
import numpy as np

rng = np.random.default_rng(42)
g30 = df.loc[df["version"] == "gate_30", "retention_7"].to_numpy()
g40 = df.loc[df["version"] == "gate_40", "retention_7"].to_numpy()

boot_diffs = []
for _ in range(2000):
    s30 = rng.choice(g30, size=g30.size, replace=True).mean()
    s40 = rng.choice(g40, size=g40.size, replace=True).mean()
    boot_diffs.append((s40 - s30) * 100)

boot_diffs = np.array(boot_diffs)
low, high = np.percentile(boot_diffs, [2.5, 97.5])
print("mean diff:", round(boot_diffs.mean(), 2), "pp | 95% CI:", round(low, 2), "to", round(high, 2), "pp")

mean diff: -0.83 pp | 95% CI: -1.35 to -0.34 pp


## 6. Power and minimum detectable effect (MDE)

The smallest true difference this experiment could detect with 80% power at α = 0.05, given the control baseline and the actual group sizes. This describes the experiment's sensitivity, not the gate's effect.

**Result:** MDE is 0.73 pp for D7 and 0.93 pp for D1. The observed D7 drop (0.82 pp) is just above the threshold. The observed D1 drop (0.59 pp) is below it, so the non-significant D1 result does not show that D1 was unaffected.

In [11]:
from scipy.stats import norm

alpha, power = 0.05, 0.80
z_alpha = norm.ppf(1 - alpha / 2)
z_power = norm.ppf(power)
n30, n40 = n["gate_30"], n["gate_40"]

for col in ["retention_7", "retention_1"]:
    p0 = df.loc[df["version"] == "gate_30", col].mean()
    se = np.sqrt(p0 * (1 - p0) * (1 / n30 + 1 / n40))
    mde_pp = (z_alpha + z_power) * se * 100
    print(col, "| baseline:", round(p0 * 100, 2), "% | MDE:", round(mde_pp, 2),
          "pp | relative MDE:", round(mde_pp / (p0 * 100) * 100, 1), "%")

retention_7 | baseline: 19.02 % | MDE: 0.73 pp | relative MDE: 3.8 %
retention_1 | baseline: 44.82 % | MDE: 0.93 pp | relative MDE: 2.1 %


## 7. Guardrail: game rounds in the first 14 days

Compares engagement volume between groups, excluding the outlier and focusing on the median because the distribution is heavily right-skewed.

**Result:** median 17 vs 16 rounds; mean 51.34 vs 51.30. No meaningful difference.

In [12]:
mask = df["userid"] != 6390605
rounds = df[mask].groupby("version")["sum_gamerounds"]
print("median:\n", rounds.median())
print("mean:\n", rounds.mean().round(2))

median:
 version
gate_30    17.0
gate_40    16.0
Name: sum_gamerounds, dtype: float64
mean:
 version
gate_30    51.34
gate_40    51.30
Name: sum_gamerounds, dtype: float64


## 8. D7 retention by engagement band (descriptive)

Splits players by 14-day game rounds (0, 1–29, 30–99, 100+) and compares D7 retention within each band.

**Result:** the gap is concentrated in the 30–99 band (27.84% vs 25.67%, −2.17 pp). Other bands differ by 0.35 pp or less.

**Caveat:** game rounds are measured after assignment and may themselves be affected by the gate position, so players in the same band are not directly comparable across groups. Rounds are also not levels. This is a lead for a follow-up test, not a causal estimate for any segment.

In [13]:
bins = [-1, 0, 29, 99, df["sum_gamerounds"].max()]
labels = ["0", "1-29", "30-99", "100+"]
df["rounds_band"] = pd.cut(df["sum_gamerounds"], bins=bins, labels=labels)

seg = df.groupby(["rounds_band", "version"], observed=True)["retention_7"].agg(["count", "mean"])
seg["mean"] = (seg["mean"] * 100).round(2)
seg

count   mean
rounds_band version              
0           gate_30   1937   0.83
            gate_40   2057   0.63
1-29        gate_30  26107   4.51
            gate_40  26819   4.18
30-99       gate_30  10488  27.84
            gate_40  10265  25.67
100+        gate_30   6168  71.14
            gate_40   6348  71.03

## 9. Summary and recommendation

**Result.** Moving the first gate from level 30 to level 40 reduced D7 retention from 19.02% to 18.20%, a drop of 0.82 percentage points (95% CI: 0.34 to 1.35 pp; p = 0.0016). The drop is larger than the experiment's minimum detectable effect (0.73 pp), and engagement volume (game rounds) did not change. D1 retention also fell (−0.59 pp), but the difference was not significant and is smaller than what this test could reliably detect.

**Recommendation.** Keep the first gate at level 30. Do not roll out `gate_40`.

**Limitations.**
- **No revenue data.** The gate is a monetization point, and this dataset does not show whether moving it changed purchases. If `gate_40` increased revenue, the trade-off would need to be re-evaluated.
- **Borderline sample ratio.** The SRM check (p = 0.0086) did not cross the pre-set 0.001 threshold, but the intended allocation is not documented.
- **Short horizon.** Only D1, D7, and 14-day game rounds are available, so longer-term retention (for example D30) is unknown.

**Next step.** The D7 gap was concentrated among players with 30–99 rounds. A follow-up test could target players near the first gate, for example by testing an intermediate gate position or a reminder at that point.